# Laya full-data training — Kaggle T4 x2

Private model training handoff. Enable Internet, GPU T4 x2 and the HF_TOKEN
Kaggle Secret. Set the private backup repository below. First run preflight,
then restore it in a fresh session, then select train. See docs/training-handoff.md.
The older execution.ipynb is the historical TPU translation workflow.


In [ ]:
CODE_REVISION = "0fa1a580758da9e020075b0537ace4166a03b328"
BACKUP_REPO = "Farahani/kodoom"  # private model repository
ACTION = "preflight"  # preflight | restore-preflight | train
RESTORE_REVISION = ""  # complete checkpoint commit from preflight summary
RESTORE_RUN_ID = "laya-full-v1-preflight"
CHECKOUT = "/kaggle/working/kodum-training-code"

## Bootstrap the pinned training code

Only package installation and orchestration belong here. GPU training runs in
two child processes; the notebook kernel does not own the model.


In [ ]:
import importlib.util
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

assert len(CODE_REVISION) == 40, "Set the pinned training commit"
assert BACKUP_REPO, "Set BACKUP_REPO to your private model repository"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
runtime_path = Path("/kaggle/working/kodoom_training_runtime.py")
runtime_path.write_bytes(
    urllib.request.urlopen(
        f"https://raw.githubusercontent.com/Farahani1/kodum/{CODE_REVISION}/src/kodoom/runtime.py",
        timeout=60,
    ).read()
)
spec = importlib.util.spec_from_file_location("kodoom_training_runtime", runtime_path)
runtime = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runtime)
checkout = runtime.bootstrap(
    "kaggle", CODE_REVISION, CHECKOUT, backend="torch", install_dependencies=False
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        f"{checkout}[train]",
        "-c",
        str(checkout / "constraints/gpu.txt"),
    ],
    check=True,
)

## Run the selected stage

Preflight and full training use different run IDs and original-Laya initialization.
Preflight does two updates, validation and a private durable checkpoint upload.
Run restore-preflight in a fresh session before switching to train. No final
test evaluation or public publication happens automatically.

Preflight succeeds only when the final summary reports preflight_complete, step 2 and a nonempty durable_revision. An error in an earlier cell means the attempt failed even if a later cell runs.


In [ ]:
from kodoom.training.launch import run

result = run(
    ACTION,
    backup_repo=BACKUP_REPO,
    restore_revision=RESTORE_REVISION,
    restore_run_id=RESTORE_RUN_ID,
)
print(result)